Code to import dataset and train ML classifier model for EMG data

In [ ]:
# Dependencies

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, GridSearchCV, KFold


#### Define Hyperparamters

In [ ]:
batch_size = 256     # number of independent sequences that will be processed in parallel for each forward/backward pass
n_epochs = 1000
K = 50  # Number of folds for cross-fitting
lr = 0.001
nfold = 50
device = 'cuda' if torch.cuda.is_available() else 'cpu'
hidden_dim = [20, 20]  # number of hidden units in each layer
dropout = 0         # dropout rate, not necessary for small NN

print(f"Using device: {device}")

#### Data Preparation

In [ ]:
df = pd.read_csv('DATAFILE.csv')

Xtr = df.drop(columns=['Threshold', 'Time', 'Classifier'])

#### Neural Network

In [ ]:
# Ridge regression with cross-validation to find the best alpha

class RidgeCV:
    def __init__(self, alphas, cv=5):
        self.alphas = alphas
        self.cv = cv
        self.best_alpha_ = None
        self.coef_ = None

    def fit(self, X, y):
        ridge_cv = RidgeCV(alphas=self.alphas, cv=self.cv)
        ridge_cv.fit(X, y)
        self.best_alpha_ = ridge_cv.alpha_
        self.coef_ = ridge_cv.coef_

    def predict(self, X):
        if self.coef_ is None:
            raise ValueError("Model is not fitted yet.")
        return X @ self.coef_

In [ ]:
# CNN Classifier Definition

class CNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, hidden_dim[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_dim[0], hidden_dim[1], kernel_size=3, padding=1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim[1], output_dim)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.adaptive_avg_pool1d(x, 1).squeeze(2)  # Global average pooling
        x = self.dropout(x)
        x = self.fc(x)
        return x